# W1-1. Workshop: 엑셀 데이터 분석 Agent (프로젝트 관리)

Labs 01, 02 복습 적용

---

이 워크숍에서는 Labs에서 학습한 LangChain Tool 정의, LangGraph Agent 구성 기술을 **프로젝트 관리 엑셀 데이터 분석**이라는 실무 시나리오에 적용합니다. pandas 기반 분석 함수를 `@tool` 데코레이터로 감싸 AI Agent가 자연어 질문에 데이터 기반으로 응답하는 End-to-End 파이프라인을 구현합니다.

### 학습 목표

1. pandas 기반 데이터 분석 Tool을 **`@tool` 데코레이터**로 정의할 수 있다
2. 엑셀 데이터를 분석하는 **AI Agent**를 `create_agent`로 구현할 수 있다
3. 자연어 질문을 **데이터 분석**으로 변환하는 흐름을 이해할 수 있다

### 진행 방식

- 각 섹션의 코드를 **순서대로 실행**하며 Agent를 완성합니다
- `# [핵심]`, `# [주의]` 주석을 읽고 동작 원리를 이해합니다
- **섹션 5**에서 3개의 테스트 질문으로 Agent 응답을 직접 확인합니다
- **섹션 6 도전 과제**에서 Tool을 확장하고 복합 분석을 실험합니다



## 1) 환경 설정
> 진행: [▓░░░░░] 1/6


In [1]:
# [W1A-1] : 라이브러리 설치
# [핵심] 엑셀 Agent 파이프라인에 필요한 모든 패키지 — 첫 실행 시 한 번만 필요
# [주의] 버전 충돌 시 런타임 재시작 후 재실행
!uv pip install -q langchain langchain-core langchain-openai langgraph pandas openpyxl python-dotenv tabulate

In [2]:
# [W1A-2] : 환경변수 로드 및 API 키 검증
# [핵심] .env 파일에서 API 키를 로드 — 하드코딩 금지 원칙
# [주의] override=True → 셀 재실행 시 .env 값으로 덮어씀

import os
from dotenv import load_dotenv

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "OPENAI_API_KEY가 .env에 설정되어 있지 않습니다."
print(f"API 키 로드 완료: {OPENAI_API_KEY[:8]}...")

API 키 로드 완료: sk-proj-...


In [3]:
# [W1A-3] : LLM 초기화
# [핵심] temperature=0 → 데이터 분석은 결정적 출력이 중요
# [주의] gpt-4.1-mini는 비용 대비 성능이 우수한 모델 — 실습 환경에 적합

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)  # 결정적 출력

print(f"모델: {llm.model_name}")

모델: gpt-4.1-mini


## 2) 시나리오 & 데이터 준비

### 시나리오

> 당신은 PMO(프로젝트 관리 오피스) 소속입니다. 경영진이 30개 진행 중인 프로젝트의 현황을 자연어로 질문하면 즉시 데이터 기반 답변을 제공하는 AI 어시스턴트를 요청했습니다.

| 항목 | 내용 |
|------|------|
| **파일** | `sample_data/pm/pm_projects.xlsx` |
| **시트** | `프로젝트현황`, `월별실적`, `리소스배분` |
| **규모** | 30개 프로젝트, 3개 시트 |
> 진행: [▓▓░░░░] 2/6


In [4]:
# [W1A-4] : 엑셀 데이터 로드 — 3개 시트
# [핵심] 시트별로 DataFrame을 분리해 로드 → 각 Tool에서 독립 접근
# [주의] sheet_name 값은 엑셀 파일의 실제 시트명과 정확히 일치해야 함

import pandas as pd

DATA_PATH = "sample_data/pm/pm_projects.xlsx"

df_projects  = pd.read_excel(DATA_PATH, sheet_name="프로젝트현황")   # 프로젝트 기본 정보
df_monthly   = pd.read_excel(DATA_PATH, sheet_name="월별실적")        # 월별 실적
df_resources = pd.read_excel(DATA_PATH, sheet_name="리소스배분")      # 리소스 배분

print(f"프로젝트현황: {df_projects.shape[0]}행 x {df_projects.shape[1]}열")
print(f"월별실적:     {df_monthly.shape[0]}행 x {df_monthly.shape[1]}열")
print(f"리소스배분:   {df_resources.shape[0]}행 x {df_resources.shape[1]}열")
print("\n[프로젝트현황 컬럼]:", df_projects.columns.tolist())
print("[월별실적 컬럼]:",     df_monthly.columns.tolist())
print("[리소스배분 컬럼]:",   df_resources.columns.tolist())

프로젝트현황: 30행 x 11열
월별실적:     120행 x 6열
리소스배분:   50행 x 5열

[프로젝트현황 컬럼]: ['project_id', 'project_name', 'department', 'manager', 'start_date', 'end_date', 'budget_million', 'spent_million', 'progress_pct', 'status', 'risk_level']
[월별실적 컬럼]: ['project_id', 'month', 'planned_tasks', 'completed_tasks', 'issues_count', 'man_hours']
[리소스배분 컬럼]: ['project_id', 'member_name', 'role', 'allocation_pct', 'skill_set']


In [5]:
# [W1A-5] : 데이터 기본 탐색
# [핵심] 컬럼명·값 범위를 파악해야 Tool에서 정확한 필터 조건 설정 가능
# [주의] 컬럼명은 Tool 코드에서 그대로 사용 — 오타 없이 정확히 확인

print("=== 프로젝트현황 (앞 3행) ===")
display(df_projects.head(3))

print("\n=== status 분포 ===")
print(df_projects["status"].value_counts())

print("\n=== 월별실적 (앞 3행) ===")
display(df_monthly.head(3))

print("\n=== 리소스배분 (앞 3행) ===")
display(df_resources.head(3))

=== 프로젝트현황 (앞 3행) ===


,project_id,project_name,department,manager,start_date,end_date,budget_million,spent_million,progress_pct,status,risk_level
0,PRJ-001,ERP 시스템 고도화,경영지원팀,김민준,2025-02-19,2025-04-28,261,139,70,보류,중간
1,PRJ-002,AI 챗봇 플랫폼 구축,인프라팀,이서연,2025-03-21,2025-04-06,434,201,100,완료,높음
2,PRJ-003,데이터 레이크 구성,경영지원팀,박지호,2025-01-06,2025-02-12,304,301,45,진행중,낮음



=== status 분포 ===
status
진행중    14
완료      8
지연      5
보류      3
Name: count, dtype: int64

=== 월별실적 (앞 3행) ===


,project_id,month,planned_tasks,completed_tasks,issues_count,man_hours
0,PRJ-001,2025-01,17,11,0,252
1,PRJ-001,2025-02,13,11,0,90
2,PRJ-001,2025-03,14,2,3,224



=== 리소스배분 (앞 3행) ===


,project_id,member_name,role,allocation_pct,skill_set
0,PRJ-004,강지은,인프라엔지니어,88,"기획, PPT, Notion"
1,PRJ-024,조태양,디자이너,29,"Java, Spring, Kubernetes"
2,PRJ-004,윤하은,인프라엔지니어,91,"Spark, Airflow, dbt"


## 3) 분석 Tool 정의 — `@tool` + pandas

`@tool` 데코레이터로 함수를 LangChain Tool로 변환합니다.

```
자연어 질문 → Agent(LLM) → Tool 선택 → pandas 분석 → 결과 반환 → LLM 답변
```

> **Tool 설계 원칙**
> - **docstring이 Tool 설명**이 됩니다 — LLM은 docstring을 읽고 Tool을 선택합니다
> - 선택적 파라미터(`= None`)로 유연한 필터링 지원
> - 반환값은 **문자열**로 — LLM이 읽을 수 있는 텍스트 형식
> 진행: [▓▓▓░░░] 3/6


In [6]:
# [W1A-6] : Tool 1 — 프로젝트 현황 조회
# [핵심] @tool 데코레이터가 함수를 LangChain Tool 객체로 변환 — docstring이 LLM 설명이 됨
# [주의] 반환값은 반드시 str — LLM에게 전달되는 context이므로 가독성 있게 포맷

from langchain_core.tools import tool

@tool
def get_project_summary(status: str = None, department: str = None) -> str:
    """프로젝트 현황을 조회합니다. status나 department로 필터링할 수 있습니다.
    
    Args:
        status: 프로젝트 상태 필터 (예: '진행중', '지연', '완료')
        department: 부서명 필터 (예: 'IT개발팀', '반도체연구소')
    """
    df = df_projects.copy()               # 원본 보호
    if status:     df = df[df["status"] == status]         # 상태 필터
    if department: df = df[df["department"] == department]  # 부서 필터

    summary  = f"총 {len(df)}개 프로젝트\n"
    summary += f"평균 진행률: {df['progress_pct'].mean():.1f}%\n"
    summary += f"총 예산: {df['budget_million'].sum()}백만원, 총 집행: {df['spent_million'].sum()}백만원\n\n"
    summary += df[["project_id","project_name","status","progress_pct","budget_million"]].to_markdown(index=False)
    return summary

In [7]:
# [W1A-7] : Tool 2 — 월별 실적 분석
# [핵심] project_id와 month 파라미터로 세부 드릴다운 가능
# [주의] 완료율 계산 시 0 나누기 방지 — total_p가 0인 경우 처리

@tool
def analyze_monthly_performance(project_id: str = None, month: str = None) -> str:
    """월별 실적을 분석합니다. 계획 대비 완료율, 이슈 발생 건수를 확인합니다.
    
    Args:
        project_id: 프로젝트 ID 필터 (예: 'PRJ-001')
        month: 월 필터 (예: '2025-03')
    """
    df = df_monthly.copy()
    if project_id: df = df[df["project_id"] == project_id]
    if month:      df = df[df["month"] == month]

    total_p = df['planned_tasks'].sum()
    total_c = df['completed_tasks'].sum()
    rate = total_c / total_p * 100 if total_p > 0 else 0.0  # 0 나누기 방지

    summary  = f"레코드: {len(df)}개\n"
    summary += f"계획: {total_p}, 완료: {total_c}, 완료율: {rate:.1f}%\n"
    summary += f"이슈: {df['issues_count'].sum()}건\n\n"
    summary += df.to_markdown(index=False)
    return summary

In [8]:
# [W1A-8] : Tool 3 — 리소스 배분 조회
# [핵심] 프로젝트별·역할별 인력 배분 현황을 조회 — 과부하 인원 파악에 활용
# [주의] role 필터는 정확한 직책명 일치 필요 — 부분 문자열 검색이 아님

@tool
def get_resource_allocation(project_id: str = None, role: str = None) -> str:
    """리소스 배분 현황을 조회합니다. 인력 할당률과 프로젝트별 투입 현황을 확인합니다.
    
    Args:
        project_id: 프로젝트 ID 필터 (예: 'PRJ-001')
        role: 직책 필터 (예: 'PM', '개발자', 'QA')
    """
    df = df_resources.copy()
    if project_id: df = df[df["project_id"] == project_id]
    if role:       df = df[df["role"] == role]

    summary  = f"총 {len(df)}명, 평균 할당률: {df['allocation_pct'].mean():.1f}%\n\n"
    summary += df.to_markdown(index=False)
    return summary

## 4) Agent 구성 — `create_agent`

**ReAct(Reasoning + Acting)** 패턴으로 동작하는 Agent를 구성합니다.

```
질문 → [Thought] 어떤 Tool을 쓸까? → [Action] Tool 호출
     → [Observation] 결과 확인    → [Thought] 충분한가?
     → ... (반복) → [Final Answer] 최종 답변
```
> 진행: [▓▓▓▓░░] 4/6


In [ ]:
# [W1A-9] : Agent 구성 — create_agent
# [핵심] create_agent(model, tools, system_prompt) — 세 인자만으로 Agent 완성
# [주의] prompt(system message)는 Agent의 페르소나와 행동 방침을 정의

from langchain.agents import create_agent

excel_agent = create_agent(
    model=llm,
    tools=[get_project_summary, analyze_monthly_performance, get_resource_allocation],
    system_prompt="""당신은 프로젝트 관리 데이터 분석 전문가입니다.
    주어진 도구를 활용하여 정확한 데이터 기반 답변을 제공하세요.
    숫자는 구체적으로 인용하고, 필요 시 여러 Tool을 순차 호출해 교차 검증하세요.""",
)

## 5) 질의 테스트 & 결과 검증

3가지 유형의 질문으로 Agent가 올바른 Tool을 선택하고 데이터 기반 답변을 생성하는지 확인합니다.

| # | 질문 유형 | 기대 Tool |
|---|-----------|----------|
| 1 | 상태 필터 조회 | `get_project_summary` |
| 2 | 예산 집행률 분석 | `get_project_summary` |
| 3 | 월별 이슈 분석 | `analyze_monthly_performance` |
> 진행: [▓▓▓▓▓░] 5/6


In [10]:
# [W1A-10] : 질의 1 — 지연 프로젝트 현황
# [핵심] invoke 결과의 messages 리스트 마지막 항목이 최종 답변
# [주의] ToolMessage가 중간에 포함 — [-1]로 최종 AIMessage만 추출

result1 = excel_agent.invoke({"messages": [{"role": "user", "content": "지연 중인 프로젝트는 몇 개이고, 어떤 프로젝트인가요?"}]})
print(result1["messages"][-1].content)

현재 지연 중인 프로젝트는 총 5개입니다. 
프로젝트 목록은 다음과 같습니다.
1. PRJ-004 보안 취약점 점검 (진행률 16%, 예산 85백만원)
2. PRJ-005 클라우드 마이그레이션 (진행률 38%, 예산 394백만원)
3. PRJ-006 고객 포털 리뉴얼 (진행률 24%, 예산 308백만원)
4. PRJ-009 DevOps 파이프라인 구축 (진행률 63%, 예산 299백만원)
5. PRJ-020 데이터 거버넌스 수립 (진행률 50%, 예산 174백만원)


In [11]:
# [W1A-11] : 질의 2 — 부서별 예산 집행률
# [핵심] department 필터 + 예산 계산 — Agent가 Tool 결과를 가공해 답변 생성
# [주의] Tool 반환값이 복잡할수록 LLM의 요약 능력이 중요

result2 = excel_agent.invoke({"messages": [{"role": "user", "content": "IT개발팀의 총 예산 대비 집행률은?"}]})
print(result2["messages"][-1].content)

IT개발팀의 총 예산은 1,231백만원이고, 총 집행액은 1,479백만원입니다. 따라서 총 예산 대비 집행률은 약 120.1%입니다. (1479 / 1231 * 100)


In [12]:
# [W1A-12] : 질의 3 — 월별 이슈 분석
# [핵심] analyze_monthly_performance Tool 활용 — month 필터로 특정 월 조회
# [주의] 여러 Tool을 조합해야 할 때 Agent가 순서대로 자동 호출

result3 = excel_agent.invoke({"messages": [{"role": "user", "content": "3월에 이슈가 가장 많았던 프로젝트는?"}]})
print(result3["messages"][-1].content)

3월에 이슈가 가장 많았던 프로젝트는 PRJ-003과 PRJ-009로 각각 9건의 이슈가 발생했습니다.


In [13]:
# [W1A-13] : 3개 질문 일괄 테스트
# [핵심] 다양한 질문 유형으로 Tool 선택 정확도 확인

test_questions = [
    "지연 중인 프로젝트는 몇 개이고, 어떤 프로젝트인가요?",
    "IT개발팀의 총 예산 대비 집행률은?",
    "3월에 이슈가 가장 많았던 프로젝트는?",
]

for i, q in enumerate(test_questions, 1):
    result = excel_agent.invoke({"messages": [{"role": "user", "content": q}]})
    answer = result["messages"][-1].content
    print(f"[Q{i}] {q}")
    print(f"[A{i}] {answer}")
    print("-" * 80)

[Q1] 지연 중인 프로젝트는 몇 개이고, 어떤 프로젝트인가요?
[A1] 현재 지연 중인 프로젝트는 총 5개입니다. 프로젝트 목록은 다음과 같습니다.

1. PRJ-004 보안 취약점 점검 (진행률 16%, 예산 85백만원)
2. PRJ-005 클라우드 마이그레이션 (진행률 38%, 예산 394백만원)
3. PRJ-006 고객 포털 리뉴얼 (진행률 24%, 예산 308백만원)
4. PRJ-009 DevOps 파이프라인 구축 (진행률 63%, 예산 299백만원)
5. PRJ-020 데이터 거버넌스 수립 (진행률 50%, 예산 174백만원)
--------------------------------------------------------------------------------
[Q2] IT개발팀의 총 예산 대비 집행률은?
[A2] IT개발팀의 총 예산은 1,231백만원이고, 총 집행액은 1,479백만원입니다. 따라서 총 예산 대비 집행률은 약 120.1%입니다. (1479 / 1231 * 100)
--------------------------------------------------------------------------------
[Q3] 3월에 이슈가 가장 많았던 프로젝트는?
[A3] 3월에 이슈가 가장 많았던 프로젝트는 PRJ-003과 PRJ-009로 각각 9건의 이슈가 발생했습니다.
--------------------------------------------------------------------------------


In [14]:
# [W1A-14] : 결과 검증 — pandas 직접 계산과 비교
# [핵심] Agent 답변을 pandas 직접 계산 결과와 대조 — 데이터 정확도 확인
# [주의] Agent 답변이 수치와 다를 경우 Tool 반환 포맷 또는 docstring 수정

# 지연 프로젝트 직접 계산
delayed = df_projects[df_projects["status"] == "지연"]
print(f"[검증 1] 지연 프로젝트 수: {len(delayed)}개")
print(delayed[["project_id", "project_name", "progress_pct"]].to_string(index=False))

print()

# IT개발팀 집행률 직접 계산
it_df = df_projects[df_projects["department"] == "IT개발팀"]
rate  = it_df["spent_million"].sum() / it_df["budget_million"].sum() * 100
print(f"[검증 2] IT개발팀 집행률: {rate:.1f}%")

[검증 1] 지연 프로젝트 수: 5개
project_id    project_name  progress_pct
   PRJ-004       보안 취약점 점검            16
   PRJ-005     클라우드 마이그레이션            38
   PRJ-006       고객 포털 리뉴얼            24
   PRJ-009 DevOps 파이프라인 구축            63
   PRJ-020     데이터 거버넌스 수립            50

[검증 2] IT개발팀 집행률: 120.1%


---

## 정리

| 항목 | 내용 |
|------|------|
| **다룬 기술** | `@tool` 데코레이터, `create_agent`, `pandas`, `openpyxl` |
| **핵심 개념** | ReAct Agent — 자연어 질문 → Tool 선택 → 데이터 분석 → 답변 생성 |
| **구현 결과** | 3개 Tool 기반 프로젝트 관리 AI Agent (질의응답 + 리포트 생성) |

**엑셀 Agent의 확장 방향**
- Tool 추가 — 예측 분석, 이상 탐지, 자동 알림
- 멀티 파일 지원 — 여러 엑셀 파일을 동적으로 로드
- 메모리 추가 — 대화 이력 기반 맥락 유지 분석

